In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import folium
import base64
from pathlib import Path

import os
# import cv2
# import json
# import onnxruntime as ort
# import seaborn as sns
# from pathlib import Path
# from itertools import chain
# from tqdm.notebook import tqdm
# from PIL import Image
# import numpy as np
# from sklearn.cluster import DBSCAN
# import folium
# import matplotlib.cm as cm
# import matplotlib.colors as mcolors
# from pyproj import Transformer
# from collections import defaultdict

from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

In [ ]:
photo_dir = r'E:\data\202502_signboard\data_annotation\field_data\0925_survey'

In [ ]:
def get_gps_from_exif(img_path):
    img = Image.open(img_path)
    exif_data = img._getexif()
    if not exif_data:
        return None
    
    gps_info = {}
    for tag, value in exif_data.items():
        decoded = TAGS.get(tag, tag)
        if decoded == "GPSInfo":
            for t in value:
                sub_decoded = GPSTAGS.get(t, t)
                gps_info[sub_decoded] = value[t]

    def convert_to_degrees(value):
        d, m, s = [float(x) for x in value]  # 🔑 这里做了修正
        return d + (m / 60.0) + (s / 3600.0)

    if "GPSLatitude" in gps_info and "GPSLongitude" in gps_info:
        lat = convert_to_degrees(gps_info["GPSLatitude"])
        if gps_info.get("GPSLatitudeRef") in ["S", "南纬"]:
            lat = -lat
        lon = convert_to_degrees(gps_info["GPSLongitude"])
        if gps_info.get("GPSLongitudeRef") in ["W", "西经"]:
            lon = -lon
        return lat, lon
    return None


In [ ]:


def img_to_base64(path, max_size=(200, 200)):
    """把图像缩放成缩略图后转 base64"""
    from PIL import Image
    img = Image.open(path)
    img.thumbnail(max_size)  # 缩略图
    from io import BytesIO
    buffer = BytesIO()
    img.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

# 例子：多张照片
photo_paths = [os.path.join(photo_dir, p) for p in os.listdir(photo_dir) if p.endswith('.jpg')]

# 初始化地图（以第一张照片为中心）
lat, lon = get_gps_from_exif(photo_paths[0])
m = folium.Map(location=[lat, lon], zoom_start=16, max_zoom=22)

for path in photo_paths:
    gps = get_gps_from_exif(path)
    if gps is None:
        continue
    lat, lon = gps
    img_b64 = img_to_base64(path)
    popup_html = f"""
    <div style="width:220px">
        <img src="data:image/jpeg;base64,{img_b64}" width="200">
        <p>{Path(path).name}</p>
    </div>
    """
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=250),
        icon=folium.Icon(color="blue", icon="camera")
    ).add_to(m)

m.save("photos_map.html")
print("✅ 已生成地图：photos_map.html")


In [ ]:
import os
from pathlib import Path
import folium
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS
import base64
from io import BytesIO


# 1. 提取 EXIF GPS
def get_gps_from_exif(img_path):
    try:
        img = Image.open(img_path)
        exif_data = img._getexif()
        if not exif_data:
            return None

        gps_info = {}
        for tag, value in exif_data.items():
            decoded = TAGS.get(tag, tag)
            if decoded == "GPSInfo":
                for t in value:
                    sub_decoded = GPSTAGS.get(t, t)
                    gps_info[sub_decoded] = value[t]

        def convert_to_degrees(value):
            d, m, s = [float(x) for x in value]
            return d + (m / 60.0) + (s / 3600.0)

        if "GPSLatitude" in gps_info and "GPSLongitude" in gps_info:
            lat = convert_to_degrees(gps_info["GPSLatitude"])
            if gps_info.get("GPSLatitudeRef") in ["S", "南纬"]:
                lat = -lat
            lon = convert_to_degrees(gps_info["GPSLongitude"])
            if gps_info.get("GPSLongitudeRef") in ["W", "西经"]:
                lon = -lon
            return lat, lon
    except Exception as e:
        print(f"❌ 无法读取 {img_path}: {e}")
    return None


# 2. 转缩略图 + Base64
def img_to_base64_thumbnail(path, max_size=(80, 80)):
    try:
        img = Image.open(path)
        img.thumbnail(max_size)
        buffer = BytesIO()
        img.save(buffer, format="JPEG")
        return base64.b64encode(buffer.getvalue()).decode("utf-8")
    except Exception as e:
        print(f"❌ 生成缩略图失败 {path}: {e}")
        return None


# 3. 主函数
def photos_to_map(photo_dir, save_file="photos_map.html"):
    photo_paths = [os.path.join(photo_dir, p) for p in os.listdir(photo_dir)
                   if p.lower().endswith((".jpg", ".jpeg", ".png"))]

    gps_points = []
    for path in photo_paths:
        gps = get_gps_from_exif(path)
        if gps:
            gps_points.append((path, gps))

    if not gps_points:
        print("⚠️ 没有照片包含 GPS 信息！")
        return

    # 以第一张照片为地图中心
    first_lat, first_lon = gps_points[0][1]
    m = folium.Map(location=[first_lat, first_lon], zoom_start=16, max_zoom=22)

    for path, (lat, lon) in gps_points:
        img_b64 = img_to_base64_thumbnail(path)
        if not img_b64:
            continue

        # 用缩略图作为图标
        icon_html = f'<img src="data:image/jpeg;base64,{img_b64}" style="width:60px;height:60px;border:1px solid black;">'
        icon = folium.DivIcon(html=icon_html)

        folium.Marker(
            location=[lat, lon],
            icon=icon,
            tooltip=Path(path).name  # 鼠标悬停显示文件名
        ).add_to(m)

    m.save(save_file)
    print(f"✅ 已生成地图: {save_file}")


# -----------------
# 使用示例：
# -----------------
photos_to_map(photo_dir, "photos_map.html")
